# NeurIPS - Open Polymer Prediction 2025
---

In [ ]:
print("Installing Local Modules")
!pip install /kaggle/input/wheels/wheels/*

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
import networkx as nx
import re
from tqdm import tqdm
tqdm.pandas()

import pickle
import gc
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

from rdkit import RDLogger
from rdkit import Chem
from rdkit.Chem import Descriptors, rdmolops, AllChem, Descriptors3D
from rdkit import DataStructs
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import MACCSkeys
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
from sklearn.model_selection import KFold, cross_val_score
import xgboost as xgb
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from rdkit.Chem.rdMolDescriptors import CalcTPSA, CalcNumRotatableBonds
from rdkit.Chem.Descriptors import MolWt, MolLogP
from sklearn.feature_selection import VarianceThreshold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator, GetAtomPairGenerator, GetTopologicalTorsionGenerator

import sys
sys.path.insert(0, "/kaggle/working")

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm.auto import tqdm
print("Importing Done")

In [ ]:
print("Mounted under /kaggle/input:\n ", os.listdir("/kaggle/input"))

DATASET_SLUG = "neurips-polymer"  # your dataset
MODEL_SLUG   = "neurips-chembert-with-augmented-data"  # your model repo

DATASET_ROOT = f"/kaggle/input/{DATASET_SLUG}"
MODEL_ROOT   = f"/kaggle/input/{MODEL_SLUG}"

print("Dataset root exists?", os.path.exists(DATASET_ROOT))
print("Model root exists?", os.path.exists(MODEL_ROOT))
print("Model root contents:\n ", os.listdir(MODEL_ROOT) if os.path.exists(MODEL_ROOT) else "N/A")

# Data Preprocessing
---


In [ ]:
targets = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']

## Data Augmentation

## Generating and Integrating Molecular Properties and Descriptors for Enhanced Predictive Modeling

In [ ]:
required_descriptors = {'graph_diameter','num_cycles','avg_shortest_path','MolWt', 'LogP', 'TPSA', 'RotatableBonds', 'NumAtoms', 'SMILES'}

filters = {
    'Tg': list(set(['deg_mean', 'FractionCSP3', 'num_cycles', 'RingCount', 'HallKierAlpha', 'SMR_VSA7', 'BertzCT', 'ring_size_6', 'fr_benzene', 'NumAromaticCarbocycles', 'NumAromaticRings', 'SlogP_VSA6', 'SlogP_VSA1', 'betw_mean', 'VSA_EState6', 'BalabanJ', 'Chi4n', 'FP_446', 'PEOE_VSA14', 'Chi3n', 'AvgIpc', 'FP_489', 'Chi1', 'HeavyAtomCount', 'NumHeterocycles', 'FP_485', 'fr_bicyclic', 'SMR_VSA10', 'FP_537', 'VSA_EState2', 'FP_539', 'FP_529', 'HeavyAtomMolWt', 'LabuteASA', 'ring_size_5', 'FP_505', 'NumAmideBonds', 'MolMR', 'FP_80', 'FP_195', 'FP_310', 'fr_amide', 'FP_509', 'FP_378', 'ExactMolWt', 'MolWt', 'FP_211', 'Chi2n', 'FP_266', 'FP_379', 'FP_207', 'FP_504', 'FP_203', 'NumAtoms', 'FP_199', 'FP_519', 'FP_123', 'FP_278', 'FpDensityMorgan1', 'FP_119', 'fr_imide', 'FP_279', 'FP_223', 'betw_std', 'FP_231', 'FP_219', 'FP_251', 'NumValenceElectrons', 'FP_480', 'Chi0n', 'FP_517', 'FP_255', 'Chi0', 'FP_522', 'FP_528', 'FP_526', 'FpDensityMorgan2', 'FP_354', 'Chi1n', 'FP_459', 'FP_547', 'FP_476', 'Chi0v', 'FP_210', 'FP_516', 'FP_382', 'FP_215', 'FP_243', 'FP_521', 'FP_227', 'NumAliphaticHeterocycles', 'FP_469', 'FP_467', 'FP_342', 'FP_549', 'FP_357', 'FP_494', 'FP_194', 'FP_546', 'FP_302']).union(required_descriptors)),

    'FFV': list(set(['MolLogP', 'LogP', 'Chi3v', 'Chi2v', 'Chi4v', 'Chi4n', 'Chi3n', 'VSA_EState6', 'SMR_VSA7', 'Chi1v', 'Chi2n', 'MolMR', 'Chi1n', 'Chi0v', 'SlogP_VSA6', 'BertzCT', 'PEOE_VSA14', 'Chi0n', 'LabuteASA', 'EState_VSA8', 'BalabanJ', 'Ipc', 'Chi1', 'deg_mean', 'VSA_EState8', 'MolWt', 'ExactMolWt', 'SMR_VSA9', 'HeavyAtomMolWt', 'Chi0', 'SMR_VSA6', 'EState_VSA5', 'FpDensityMorgan3', 'Kappa1', 'AvgIpc', 'FpDensityMorgan2', 'SlogP_VSA8', 'HallKierAlpha', 'FP_39', 'avg_shortest_path', 'SMR_VSA1', 'SlogP_VSA5', 'betw_mean', 'TPSA', 'FpDensityMorgan1', 'lap_eig_6', 'qed', 'lap_eig_7', 'lap_eig_8', 'RingCount', 'NumValenceElectrons', 'NumAromaticRings', 'lap_eig_5', 'num_cycles', 'EState_VSA7', 'Kappa2', 'NumAtoms', 'ring_size_6', 'betw_std', 'lap_eig_4', 'lap_eig_3', 'HeavyAtomCount', 'fr_benzene', 'NumHDonors', 'NumAromaticCarbocycles', 'PEOE_VSA7', 'SlogP_VSA2', 'SlogP_VSA3', 'NHOHCount', 'NOCount', 'fr_NH1', 'EState_VSA4', 'FP_515', 'MaxEStateIndex', 'MaxAbsEStateIndex', 'lap_eig_2', 'PEOE_VSA6', 'VSA_EState5', 'EState_VSA6', 'FractionCSP3', 'EState_VSA3', 'Phi', 'FP_535', 'NumHAcceptors', 'SlogP_VSA4', 'fr_C_O', 'FP_446', 'FP_488', 'SMR_VSA10', 'PEOE_VSA9', 'fr_C_O_noCOO', 'FP_125', 'FP_474', 'SlogP_VSA7', 'graph_diameter', 'SlogP_VSA12', 'FP_507', 'fr_bicyclic', 'MinAbsEStateIndex', 'deg_std']).union(required_descriptors)),

    'Tc': list(set(['deg_std', 'fr_unbrch_alkane', 'FP_287', 'FP_286', 'betw_mean', 'avg_shortest_path', 'Kappa3', 'graph_diameter', 'FP_285', 'FP_187', 'FP_191', 'FP_139', 'VSA_EState7', 'FpDensityMorgan3', 'FP_143', 'FpDensityMorgan2', 'FP_171', 'Phi', 'FpDensityMorgan1', 'FP_167', 'FP_175', 'qed', 'FP_163', 'FP_131', 'FP_531', 'FP_502', 'FP_142', 'Kappa2', 'FP_135', 'FP_179', 'SlogP_VSA5', 'FP_513', 'FP_134', 'SMR_VSA5', 'FP_93', 'FP_138', 'FP_130', 'RotatableBonds', 'NumRotatableBonds', 'FP_182', 'FP_162', 'FP_518', 'FP_491', 'FP_174', 'FP_496', 'FP_284', 'FP_141', 'FP_475', 'lap_eig_7', 'FP_154', 'lap_eig_8', 'FP_512', 'FP_453', 'FP_256', 'FP_488', 'deg_max', 'FP_170', 'FP_137', 'FP_190', 'FP_17', 'FP_466', 'FP_535', 'FP_474', 'fr_NH1', 'lap_eig_6', 'Chi3n', 'FP_133', 'PEOE_VSA6', 'FP_178', 'FP_186', 'Chi1n', 'FP_450', 'FP_102', 'FP_508', 'EState_VSA5', 'NumHDonors', 'NumAtomStereoCenters', 'NumUnspecifiedAtomStereoCenters', 'NHOHCount', 'FP_183', 'Chi2n', 'Chi3v', 'FP_89', 'SPS', 'betw_max', 'AvgIpc', 'Chi4n', 'Chi1v', 'FP_478', 'lap_eig_5', 'FP_484', 'SMR_VSA3', 'FP_192', 'FP_166', 'Kappa1', 'FP_495', 'FP_526', 'fr_halogen', 'FP_153', 'FP_28']).union(required_descriptors)),

    'Density': list(set(['SMR_VSA5', 'VSA_EState8', 'VSA_EState7', 'SlogP_VSA5', 'SMR_VSA10', 'FractionCSP3', 'EState_VSA5', 'SlogP_VSA12', 'VSA_EState10', 'fr_unbrch_alkane', 'NumRotatableBonds', 'RotatableBonds', 'FP_119', 'FP_513', 'PEOE_VSA8', 'Kappa3', 'PEOE_VSA7', 'FP_180', 'FP_472', 'FP_428', 'Kappa2', 'FP_80', 'Phi', 'FP_539', 'FP_512', 'FP_531', 'EState_VSA7', 'FP_537', 'FP_502', 'FP_98', 'NumHAcceptors', 'Chi1n', 'MaxAbsEStateIndex', 'MaxEStateIndex', 'PEOE_VSA14', 'FP_500', 'MolLogP', 'LogP', 'FP_465', 'MinAbsEStateIndex', 'Chi2n', 'SlogP_VSA7', 'FP_176', 'avg_shortest_path', 'EState_VSA4', 'FP_181', 'lap_eig_5', 'Chi0n', 'HallKierAlpha', 'PEOE_VSA5', 'qed', 'graph_diameter', 'FP_186', 'betw_mean', 'FP_287', 'FP_179', 'lap_eig_4', 'FP_134', 'Chi3n', 'NOCount', 'fr_C_S', 'FP_131', 'FP_177', 'FP_166', 'FP_127', 'FP_162', 'FP_191', 'FP_143', 'Chi4n', 'TPSA', 'lap_eig_3', 'Chi1v', 'SlogP_VSA6', 'FP_178', 'FP_457', 'FP_139', 'FP_163', 'SMR_VSA7', 'SlogP_VSA11', 'SlogP_VSA3', 'FP_183', 'Chi0', 'FP_137', 'ring_size_6', 'FP_138', 'fr_benzene', 'NumAromaticCarbocycles', 'FP_420', 'NumAromaticRings', 'NumAromaticHeterocycles', 'FP_492', 'FP_169', 'FP_284', 'Chi1', 'FP_141', 'FP_35', 'FP_182', 'FP_521', 'EState_VSA3', 'FP_135']).union(required_descriptors)),

    'Rg': list(set(['FP_93', 'SlogP_VSA7', 'PEOE_VSA14', 'qed', 'FP_544', 'VSA_EState8', 'FP_499', 'SlogP_VSA1', 'fr_unbrch_alkane', 'FP_42', 'EState_VSA4', 'FP_192', 'FP_508', 'FP_520', 'lap_eig_8', 'Phi', 'FP_155', 'NumAtomStereoCenters', 'NumUnspecifiedAtomStereoCenters', 'avg_shortest_path', 'FP_17', 'FP_317', 'lap_eig_7', 'FP_73', 'VSA_EState7', 'FP_224', 'fr_ester', 'graph_diameter', 'Kappa2', 'NumAmideBonds', 'fr_NH1', 'FP_191', 'fr_amide', 'FP_286', 'Kappa3', 'FP_159', 'FP_488', 'FP_33', 'deg_std', 'FP_280', 'FP_364', 'FP_287', 'EState_VSA5', 'SlogP_VSA5', 'FP_515', 'TPSA', 'FP_151', 'SMR_VSA5', 'FP_498', 'NOCount', 'betw_mean', 'RotatableBonds', 'NumRotatableBonds', 'FP_273', 'SMR_VSA3', 'FP_163', 'FP_134', 'FP_478', 'FP_138', 'FP_187', 'FP_137', 'FP_252', 'VSA_EState3', 'FP_171', 'FP_175', 'lap_eig_6', 'NHOHCount', 'Chi4v', 'FpDensityMorgan1', 'FP_182', 'FP_526', 'FP_167', 'FP_486', 'FP_142', 'FP_316', 'AvgIpc', 'MolLogP', 'LogP', 'FP_183', 'FP_130', 'FP_102', 'FP_1', 'FP_115', 'SMR_VSA10', 'Chi4n', 'FP_24', 'FP_533', 'NumHDonors', 'FP_193', 'FP_147', 'FP_38', 'Chi3n', 'FP_249', 'FP_453', 'FP_535', 'FP_492', 'Chi3v', 'FP_240', 'FP_501', 'FP_139']).union(required_descriptors))
}

In [ ]:
def smiles_to_combined_fingerprints_with_descriptors(smiles_list, radius=2, n_bits=128):
    generator = GetMorganGenerator(radius=radius, fpSize=n_bits)
    atom_pair_gen = GetAtomPairGenerator(fpSize=n_bits)
    torsion_gen = GetTopologicalTorsionGenerator(fpSize=n_bits)

    fingerprints = []
    descriptors = []
    valid_smiles = []
    invalid_indices = []

    for i, smiles in tqdm(enumerate(smiles_list), total=len(smiles_list), desc="🔬 Data Augmentation"):
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            # Fingerprints
            morgan_fp = generator.GetFingerprint(mol)
            atom_pair_fp = atom_pair_gen.GetFingerprint(mol)
            torsion_fp = torsion_gen.GetFingerprint(mol)
            maccs_fp = MACCSkeys.GenMACCSKeys(mol)

            combined_fp = np.concatenate([
                np.array(morgan_fp),
                np.array(atom_pair_fp),
                np.array(torsion_fp),
                np.array(maccs_fp)
            ])
            fingerprints.append(combined_fp)

            # RDKit Descriptors
            descriptor_values = {}
            for name, func in Descriptors.descList:
                try:
                    descriptor_values[name] = func(mol)
                except:
                    descriptor_values[name] = None

            # Specific descriptors
            descriptor_values['MolWt'] = MolWt(mol)
            descriptor_values['LogP'] = MolLogP(mol)
            descriptor_values['TPSA'] = CalcTPSA(mol)
            descriptor_values['RotatableBonds'] = CalcNumRotatableBonds(mol)
            descriptor_values['NumAtoms'] = mol.GetNumAtoms()
            descriptor_values['SMILES'] = smiles

            # Graph-based features
            try:
                adj = rdmolops.GetAdjacencyMatrix(mol)
                G = nx.from_numpy_array(adj)

                if nx.is_connected(G):
                    descriptor_values['graph_diameter'] = nx.diameter(G)
                    descriptor_values['avg_shortest_path'] = nx.average_shortest_path_length(G)
                else:
                    descriptor_values['graph_diameter'] = 0
                    descriptor_values['avg_shortest_path'] = 0

                cycles = nx.cycle_basis(G)
                descriptor_values['num_cycles'] = len(list(cycles))
                sizes = [len(c) for c in cycles]
                for k in range(3, 9):
                    descriptor_values[f'ring_size_{k}'] = sizes.count(k)
            except:
                descriptor_values['graph_diameter'] = None
                descriptor_values['avg_shortest_path'] = None
                descriptor_values['num_cycles'] = None
                for k in range(3, 9):
                    descriptor_values[f'ring_size_{k}'] = None
                    
            # Compute Centralities
            adj = rdmolops.GetAdjacencyMatrix(mol)
            G = nx.from_numpy_array(adj)
            deg = dict(nx.degree(G))
            bc = nx.betweenness_centrality(G)
            cc = nx.clustering(G)
            for label, metric in [('deg', deg), ('betw', bc), ('clust', cc)]:
                vals = np.array(list(metric.values()), dtype=float)
                descriptor_values[f'{label}_mean'] = vals.mean()
                descriptor_values[f'{label}_std'] = vals.std()
                descriptor_values[f'{label}_max'] = vals.max()
                    
            # Compute Spectral
            adj = rdmolops.GetAdjacencyMatrix(mol)
            G = nx.from_numpy_array(adj)
            L = nx.normalized_laplacian_matrix(G).toarray()
            eigs = np.linalg.eigvals(L)
            eigs = np.sort(eigs.real)
            for i in range(min(k, len(eigs))):
                descriptor_values[f'lap_eig_{i+1}'] = eigs[i]
            for i in range(len(eigs), k):
                descriptor_values[f'lap_eig_{i+1}'] = 0.0

            descriptor_values['Ipc'] = np.log10(descriptor_values['Ipc'])
            descriptor_values['lap_eig_1'] = np.sign(descriptor_values['lap_eig_1']) * np.log10(np.abs(descriptor_values['lap_eig_1']) + 1e-20)
            
            ###
            descriptors.append(descriptor_values)
            valid_smiles.append(smiles)
        else:
            fingerprints.append(np.zeros(n_bits * 3 + 167))
            descriptors.append(None)
            valid_smiles.append(None)
            invalid_indices.append(i)

    return np.array(fingerprints), descriptors, valid_smiles, invalid_indices

----

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import glob,os
import pandas as pd
import deepchem as dc
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw, PyMol, rdFMCS
from rdkit.Chem.Draw import IPythonConsole
from rdkit import rdBase
from deepchem import metrics
from IPython.display import Image, display
from rdkit.Chem.Draw import SimilarityMaps
import tensorflow as tf

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import math
import time
import random
from collections import defaultdict
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
import joblib
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

# PyG imports
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

# MyGNN Model Inference
---


In [ ]:
from tqdm.auto import tqdm
import math

class DeepChemTqdmCallback:
    """
    DeepChem-style callback: called as callback(model, current_step).
    Shows a per-epoch tqdm bar (updates once per batch).
    """
    def __init__(self, dataset, batch_size, leave=False):
        self.dataset = dataset
        self.batch_size = int(batch_size)
        self.leave = leave
        # Try to infer dataset length
        try:
            self.n = len(dataset)
        except Exception:
            y = getattr(dataset, "y", None)
            if hasattr(y, "shape"):
                self.n = int(y.shape[0])
            else:
                self.n = None
        self.steps = None if self.n is None else math.ceil(self.n / self.batch_size)
        self.pbar = None
        self.last_epoch = -1

    def __call__(self, model, current_step):
        """
        Called by DeepChem as callback(model, current_step) after each batch.
        current_step is an integer (global batch count).
        """
        # ensure int
        step = int(current_step)

        # If we can't infer steps_per_epoch, show an indeterminate progress spinner
        if self.steps is None:
            if self.pbar is None:
                self.pbar = tqdm(total=None, desc=f"Step {step}", leave=self.leave)
            else:
                self.pbar.update(1)
            return

        # Determine epoch and batch-within-epoch
        epoch = step // self.steps
        batch_in_epoch = step % self.steps

        # If new epoch, close previous bar and open a new one
        if epoch != self.last_epoch:
            if self.pbar is not None:
                try:
                    self.pbar.close()
                except Exception:
                    pass
            self.pbar = tqdm(total=self.steps, desc=f"Epoch {epoch+1}", leave=self.leave)
            self.last_epoch = epoch
            # Update bar to current batch (handles possible non-1 step jumps)
            # Usually first call in epoch will have batch_in_epoch == 0 -> update by 1
            self.pbar.update(batch_in_epoch + 1)
            return

        # Same epoch: advance by 1 (typical case)
        if self.pbar is not None:
            self.pbar.update(1)

    def close(self):
        """Call after training to ensure bar closed."""
        if self.pbar is not None:
            try:
                self.pbar.close()
            except Exception:
                pass
            self.pbar = None

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW  # CHANGED: use AdamW
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# PyG
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

def compute_weighted_score(per_target_maes: dict, sample_counts: dict):
    """
    per_target_maes: {'Tg': mae_value, 'FFV':..}
    sample_counts: {'Tg': N_tg, ...}
    weights = (1/sqrt(N)) normalized to sum 1
    """
    keys = list(per_target_maes.keys())
    inv_sqrt = np.array([1.0/math.sqrt(sample_counts[k]) for k in keys])
    weights = inv_sqrt / inv_sqrt.sum()
    score = sum(per_target_maes[k] * w for k,w in zip(keys, weights))
    return score, dict(zip(keys, weights))

ATOM_LIST = ["C","H","O","N","S","F","Cl","Br","I","P"]
MAX_DEGREE = 5
def atom_features(atom):
    at = atom.GetSymbol()
    one_hot = [1.0 if at == a else 0.0 for a in ATOM_LIST]
    if not any(one_hot): one_hot.append(1.0)
    else: one_hot.append(0.0)
    charge = [atom.GetFormalCharge()]
    aromatic = [1.0 if atom.GetIsAromatic() else 0.0]
    deg = atom.GetDegree()
    deg_oh = [1.0 if deg==d else 0.0 for d in range(MAX_DEGREE+1)]
    feats = one_hot + charge + aromatic + deg_oh
    return np.array(feats, dtype=np.float32)

def mol_to_pyg_data(smiles, global_features=None, y=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_feats = [atom_features(a) for a in mol.GetAtoms()]
    x = torch.tensor(np.vstack(node_feats), dtype=torch.float)
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        a1 = bond.GetBeginAtomIdx(); a2 = bond.GetEndAtomIdx()
        edge_index.append([a1, a2]); edge_index.append([a2, a1])
        bt = bond.GetBondType()
        bond_type = [0.0,0.0,0.0,0.0]
        if bt == Chem.rdchem.BondType.SINGLE: bond_type[0]=1.0
        elif bt == Chem.rdchem.BondType.DOUBLE: bond_type[1]=1.0
        elif bt == Chem.rdchem.BondType.TRIPLE: bond_type[2]=1.0
        elif bt == Chem.rdchem.BondType.AROMATIC: bond_type[3]=1.0
        edge_attr.append(bond_type); edge_attr.append(bond_type)

    if len(edge_index)==0:
        edge_index = [[0,0]]; edge_attr = [[0,0,0,0]]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(np.vstack(edge_attr), dtype=torch.float)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    if global_features is not None:
        gf = torch.tensor(global_features, dtype=torch.float)
        if gf.dim()==1: gf = gf.unsqueeze(0)  # shape (1, D)
        data.global_feats = gf
    else:
        data.global_feats = torch.zeros(1, dtype=torch.float)
    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)
    return data

def mol_to_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold, isomericSmiles=False)
    except Exception:
        return None

def scaffold_fold_assignments(df, n_folds=5, smiles_col="SMILES", seed=42):
    random.seed(seed)
    scaffolds = {}
    for idx, smi in enumerate(df[smiles_col].values):
        scaf = mol_to_scaffold(smi)
        if scaf is None: scaf = f"EMPTY_{idx}"
        scaffolds.setdefault(scaf, []).append(idx)
    groups = sorted(scaffolds.items(), key=lambda x: len(x[1]), reverse=True)
    fold_sizes = [0]*n_folds
    fold_assign = np.zeros(len(df), dtype=int)
    for scaf, idxs in groups:
        f = int(np.argmin(fold_sizes))
        for idx in idxs: fold_assign[idx] = f
        fold_sizes[f] += len(idxs)
    return fold_assign

class GNNWithGlobalFeats(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, global_in_dim,
                 gnn_hidden=128, n_gnn_layers=3, mlp_hidden=128, dropout=0.2,
                 conv_type='gcn'):  # NEW: conv_type
        super().__init__()
        self.global_in_dim = global_in_dim
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()  # CHANGED: BatchNorms for node features after conv
        in_dim = node_in_dim
        for _ in range(n_gnn_layers):
            if conv_type == 'gcn':
                self.convs.append(GCNConv(in_dim, gnn_hidden))
            elif conv_type == 'gat':
                # CHANGED: GAT with single head for simplicity
                self.convs.append(GATConv(in_dim, gnn_hidden // 1, heads=1, concat=False))
            else:
                raise ValueError("conv_type must be 'gcn' or 'gat'")
            self.bns.append(nn.BatchNorm1d(gnn_hidden))
            in_dim = gnn_hidden

        self.pool = global_mean_pool
        total_in = gnn_hidden + global_in_dim
        self.mlp = nn.Sequential(
            nn.Linear(total_in, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden//2, 1)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)        # CHANGED
            x = F.relu(x)
            x = self.dropout(x)
        batch = data.batch if hasattr(data, 'batch') else torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        pooled = self.pool(x, batch)
        gfeat = data.global_feats.to(pooled.dtype).to(pooled.device)
        # same robust handling as before
        if gfeat.dim() == 1:
            if gfeat.numel() == self.global_in_dim:
                gfeat = gfeat.unsqueeze(0).expand(pooled.size(0), -1)
            elif gfeat.numel() == pooled.size(0) * self.global_in_dim:
                gfeat = gfeat.view(pooled.size(0), self.global_in_dim)
            else:
                raise ValueError("Unexpected global_feats shape")
        elif gfeat.dim() == 2:
            if gfeat.size(0) != pooled.size(0) and gfeat.numel() == pooled.size(0) * self.global_in_dim:
                gfeat = gfeat.view(pooled.size(0), self.global_in_dim)
            elif gfeat.size(0) != pooled.size(0):
                gfeat = gfeat.mean(dim=0, keepdim=True).expand(pooled.size(0), -1)

        out = self.mlp(torch.cat([pooled, gfeat], dim=1))
        return out.view(-1)

def train_one_epoch(model, loader, optimizer, device, loss_fn, clip_norm=None):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred, batch.y.view(-1))
        loss.backward()
        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)  # CHANGED
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

def evaluate_mae_inverse_scaled(model, loader, device, y_scaler):
    """Evaluate MAE but invert target scaling back to original units before MAE."""
    model.eval()
    ys = []
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch).detach().cpu().numpy().tolist()
            y_batch = batch.y.view(-1).detach().cpu().numpy().tolist()
            preds.extend(pred)
            ys.extend(y_batch)
    # inverse transform (y_scaler expects 2D)
    preds = np.array(preds).reshape(-1, 1)
    ys = np.array(ys).reshape(-1, 1)
    preds_orig = y_scaler.inverse_transform(preds).ravel()
    ys_orig = y_scaler.inverse_transform(ys).ravel()
    return mean_absolute_error(ys_orig, preds_orig), ys_orig, preds_orig

def randomized_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return Chem.MolToSmiles(mol, doRandom=True)

def run_single_train_until_target(csv_path,
                                  target_col,
                                  descriptor_cols,
                                  smiles_col="SMILES",
                                  device='cuda' if torch.cuda.is_available() else 'cpu',
                                  seed=42,
                                  max_epochs=1000,
                                  patience=30,
                                  batch_size=32,
                                  conv_type='gcn',
                                  n_augment_small=1,
                                  clip_grad_norm=5.0,
                                  target_mae=None,
                                  tol_rel=0.05,
                                  tol_abs=1e-6,
                                  verbose=True):
    """
    Train one model on a single train/val split and stop early when validation MAE
    is within tolerance of `target_mae` (or when early-stopping triggers).
    Returns saved artifact paths and the achieved val MAE.
    """
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    df = pd.read_csv(csv_path)
    df = df[df[target_col].notna()].reset_index(drop=True)

    # Create scaffold folds and pick a single validation fold (fold 0)
    fold_assign = scaffold_fold_assignments(df, n_folds=5, smiles_col=smiles_col, seed=seed)
    train_idx = [i for i,f in enumerate(fold_assign) if f != 0]
    val_idx   = [i for i,f in enumerate(fold_assign) if f == 0]

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    # Fit scalers on TRAIN only
    desc_scaler = StandardScaler().fit(train_df[descriptor_cols].values.astype(float))
    y_scaler = StandardScaler().fit(train_df[[target_col]].values.astype(float))

    # Transform
    X_train = desc_scaler.transform(train_df[descriptor_cols].values.astype(float))
    X_val   = desc_scaler.transform(val_df[descriptor_cols].values.astype(float))
    y_train = y_scaler.transform(train_df[[target_col]].values.astype(float)).ravel()
    y_val   = y_scaler.transform(val_df[[target_col]].values.astype(float)).ravel()

    # Build graph objects
    train_data = []
    for i in range(len(train_df)):
        smi = train_df.loc[i, smiles_col]
        d = mol_to_pyg_data(smi, global_features=X_train[i], y=float(y_train[i]))
        if d is None:
            continue
        d.idx = train_idx[i]
        d.orig_smiles = smi
        train_data.append(d)

    # optional augmentation for small train sets
    if len(train_data) < 2000 and n_augment_small > 0:
        aug_list = []
        for d in train_data:
            for _ in range(n_augment_small):
                rs = randomized_smiles(d.orig_smiles)
                if rs is None: continue
                aug_d = mol_to_pyg_data(rs, global_features=d.global_feats.detach().cpu().numpy().ravel(), y=d.y.item())
                if aug_d is None: continue
                aug_d.idx = d.idx
                aug_d.orig_smiles = rs
                aug_list.append(aug_d)
        if aug_list:
            train_data += aug_list
            if verbose: print(f"Augmented train set with {len(aug_list)} randomized-smiles.")

    val_data = []
    for i in range(len(val_df)):
        smi = val_df.loc[i, smiles_col]
        d = mol_to_pyg_data(smi, global_features=X_val[i], y=float(y_val[i]))
        if d is None:
            continue
        d.idx = val_idx[i]
        d.orig_smiles = smi
        val_data.append(d)

    # Dataloaders
    train_loader = PyGDataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = PyGDataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=0)

    # model
    node_dim = train_data[0].x.shape[1]
    edge_dim = train_data[0].edge_attr.shape[1] if hasattr(train_data[0], 'edge_attr') else 0
    global_dim = train_data[0].global_feats.shape[0] if train_data[0].global_feats.dim()==1 else train_data[0].global_feats.shape[1]

    model = GNNWithGlobalFeats(node_in_dim=node_dim, edge_in_dim=edge_dim, global_in_dim=global_dim,
                               gnn_hidden=128, n_gnn_layers=3, mlp_hidden=128, dropout=0.2,
                               conv_type=conv_type).to(device)

    loss_fn = nn.SmoothL1Loss()
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-6)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, verbose=verbose, min_lr=1e-6)

    best_val_mae = float('inf')
    best_state = None
    no_improve = 0
    best_epoch = -1

    # target stopping thresholds
    stop_enabled = (target_mae is not None)
    if stop_enabled:
        tol = max(tol_rel * float(target_mae), tol_abs)
        if verbose:
            print(f"Target MAE {target_mae} with tolerance {tol} (relative tol {tol_rel}, abs tol {tol_abs})")

    for epoch in range(1, max_epochs+1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, loss_fn, clip_norm=clip_grad_norm)
        val_mae, ys_orig, preds_orig = evaluate_mae_inverse_scaled(model, val_loader, device, y_scaler)
        scheduler.step(val_mae)

        if val_mae < best_val_mae - 1e-6:
            best_val_mae = val_mae
            best_epoch = epoch
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"Epoch {epoch:04d} | train_loss {train_loss:.6f} | val_mae_orig {val_mae:.6f} | best_val {best_val_mae:.6f}")

        # stop if we reached target MAE within tolerance
        if stop_enabled and abs(val_mae - float(target_mae)) <= tol:
            if verbose:
                print(f"Stopping at epoch {epoch} because val_mae {val_mae:.6f} is within tolerance of target {target_mae}.")
            break

        # normal early stopping
        if no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch} (best_epoch {best_epoch} | best_val_mae {best_val_mae:.6f})")
            break

    # restore best
    if best_state is not None:
        model.load_state_dict({k: best_state[k].to(device) for k in best_state})

    # final metric
    val_mae_final, ys_orig, preds_orig = evaluate_mae_inverse_scaled(model, val_loader, device, y_scaler)
    if verbose:
        print(f"Final val MAE (orig scale): {val_mae_final:.6f} (best_epoch {best_epoch})")

    # save model + scalers (single-model artifacts)
    base = f"{target_col}_single"
    pkg = {
        'state_dict': {k: v.cpu().clone() for k, v in model.state_dict().items()},
        'node_dim': node_dim, 'edge_dim': edge_dim, 'global_dim': global_dim,
        'gnn_hidden': 128, 'n_gnn_layers': 3, 'mlp_hidden': 128, 'dropout': 0.2,
        'conv_type': conv_type, 'val_mae_orig': float(val_mae_final)
    }
    model_path = f"model_{base}.pt"
    desc_path = f"desc_scaler_{base}.pkl"
    y_path = f"y_scaler_{base}.pkl"
    joblib.dump(desc_scaler, desc_path)
    joblib.dump(y_scaler, y_path)
    torch.save(pkg, model_path)
    if verbose:
        print(f"Saved model -> {model_path}; scalers -> {desc_path}, {y_path}")

    return {
        'model_path': model_path,
        'desc_scaler_path': desc_path,
        'y_scaler_path': y_path,
        'val_mae': val_mae_final,
        'epoch': best_epoch,
        'preds_val': preds_orig,
        'y_val': ys_orig
    }

In [ ]:
%%bash
cp -r /kaggle/input/neurips-chembert-with-augmented-data/transformers/default/29/GNN_v4/* /kaggle/working/

In [ ]:
import glob, joblib, numpy as np, pandas as pd, torch
from sklearn.metrics import mean_absolute_error

def eval_on_host_train(target, host_train_csv, model_pattern="model_{}_fold*.pt", desc_cols_file=None, aggregate='mean', evaluate=False):
    # load host train
    df = pd.read_csv(host_train_csv).reset_index(drop=True)
    n = len(df)
    desc_cols = joblib.load(desc_cols_file) if desc_cols_file else [c for c in df.columns if c not in ['SMILES', target]]
    # collect models
    model_files = sorted(glob.glob(model_pattern.format(target)))
    assert model_files, "No fold models found"
    all_preds = np.zeros((len(model_files), n), dtype=float)
    for i, mp in enumerate(model_files):
        pkg = torch.load(mp, map_location='cpu')
        # load per-fold scalers (must exist)
        desc_scaler = joblib.load(pkg['scaler_files']['desc'])
        y_scaler = joblib.load(pkg['scaler_files']['y'])
        # scale whole-host descriptors using this fold's scaler
        X = df[desc_cols].values.astype(float)
        Xs = desc_scaler.transform(X)
        # build graphs for host rows (order preserved)
        data_list = []
        for idx in range(n):
            smi = df.loc[idx, 'SMILES']
            d = mol_to_pyg_data(smi, global_features=Xs[idx], y=None)
            if d is None:
                # fallback single-node graph
                from rdkit import Chem
                zero = torch.zeros((1, len(atom_features(Chem.Atom('C')))), dtype=torch.float)
                d = Data(x=zero, edge_index=torch.tensor([[0],[0]], dtype=torch.long),
                         edge_attr=torch.zeros((1,4)), global_feats=torch.tensor(Xs[idx], dtype=torch.float))
            d.orig_idx = torch.tensor(idx, dtype=torch.long)
            data_list.append(d)
        loader = PyGDataLoader(data_list, batch_size=64, shuffle=False, num_workers=0)
        # instantiate model and load weights
        model = GNNWithGlobalFeats(node_in_dim=pkg['node_dim'], edge_in_dim=pkg['edge_dim'], global_in_dim=pkg['global_dim'],
                                   gnn_hidden=pkg.get('gnn_hidden',128), n_gnn_layers=pkg.get('n_gnn_layers',3),
                                   mlp_hidden=pkg.get('mlp_hidden',128), dropout=pkg.get('dropout',0.2),
                                   conv_type=pkg.get('conv_type','gcn'))
        model.load_state_dict(pkg['state_dict'])
        model.eval()
        preds_fold = np.zeros(n, dtype=float)
        with torch.no_grad():
            for batch in loader:
                batch = batch.to('cpu')
                out = model(batch).detach().cpu().numpy()
                if hasattr(batch, 'orig_idx'):
                    idxs = batch.orig_idx.detach().cpu().numpy().ravel()
                    for p, idx in zip(out.tolist(), idxs.tolist()):
                        preds_fold[int(idx)] = p
                else:
                    # fallback sequential
                    pass
        # inverse-scale fold preds to original units
        preds_orig = y_scaler.inverse_transform(preds_fold.reshape(-1,1)).ravel()
        all_preds[i, :] = preds_orig

    # aggregate
    if aggregate == 'mean':
        final = all_preds.mean(axis=0)
    else:
        final = all_preds.mean(axis=0)  # extendable to weighted

    if evaluate:
        # compute host-train MAE
        host_mae = mean_absolute_error(subtables[target][target].values.astype(float), final)
        print(f"Host-train MAE for {target}: {host_mae:.6f} (using {len(model_files)} fold models)")
        return host_mae, final, all_preds
    else:
        return final, all_preds

---
# DA GNN Inference
---

In [ ]:
def pred_on_DA_GNN(label, model_dir, val_file):
    Restore_MODEL_DIR = model_dir

    ######################################## Featurizerization #########################
    print("# Featurizerization -> ", end="")
    featurizer = dc.feat.ConvMolFeaturizer()
    loader = dc.data.CSVLoader(tasks=[], feature_field="SMILES", featurizer=featurizer)
    testset = loader.create_dataset(val_file, shard_size=10000)
    
    ######################################## Model ######################################
    val_pred = []
    print("Predicting -> ", end="")
    for i in range(5):
        MODEL_DIR = Restore_MODEL_DIR + '/' + 'loop' + str(i + 1)
        model = dc.models.GraphConvModel(1, mode="regression", model_dir=MODEL_DIR)
        model.restore()
    
        ######################################## Predict ########################################
        val_pred.append(model.predict(testset))
    
    print("Done")
    val_pred = sum(val_pred) / len(val_pred)
    
    return val_pred

# NIPS GNN Inference
---

In [ ]:
class MolecularGraphNeuralNetwork(nn.Module):
    def __init__(self, N_fingerprints, dim, layer_hidden, layer_output):
        super(MolecularGraphNeuralNetwork, self).__init__()
        self.embed_fingerprint = nn.Embedding(N_fingerprints, dim)
        self.layer_hidden = layer_hidden
        self.layer_output = layer_output
        self.W_fingerprint = nn.ModuleList([nn.Linear(dim, dim)
                                            for _ in range(layer_hidden)])
        self.W_output = nn.ModuleList([nn.Linear(dim, dim)
                                       for _ in range(layer_output)])
        if task == 'classification':
            self.W_property = nn.Linear(dim, 2)
        if task == 'regression':
            self.W_property = nn.Linear(dim, 1)

    def pad(self, matrices, pad_value):
        """Pad the list of matrices
        with a pad_value (e.g., 0) for batch processing.
        For example, given a list of matrices [A, B, C],
        we obtain a new matrix [A00, 0B0, 00C],
        where 0 is the zero (i.e., pad value) matrix.
        """
        shapes = [m.shape for m in matrices]
        M, N = sum([s[0] for s in shapes]), sum([s[1] for s in shapes])
        zeros = torch.FloatTensor(np.zeros((M, N))).to(device)
        pad_matrices = pad_value + zeros
        i, j = 0, 0
        for k, matrix in enumerate(matrices):
            m, n = shapes[k]
            pad_matrices[i:i+m, j:j+n] = matrix
            i += m
            j += n
        return pad_matrices

    def update(self, matrix, vectors, layer):
        hidden_vectors = torch.relu(self.W_fingerprint[layer](vectors))
        return hidden_vectors + torch.matmul(matrix, hidden_vectors)

    def sum(self, vectors, axis):
        sum_vectors = [torch.sum(v, 0) for v in torch.split(vectors, axis)]
        return torch.stack(sum_vectors)

    def mean(self, vectors, axis):
        mean_vectors = [torch.mean(v, 0) for v in torch.split(vectors, axis)]
        return torch.stack(mean_vectors)

    def gnn(self, inputs):

        """Cat or pad each input data for batch processing."""
        fingerprints, adjacencies, molecular_sizes = inputs
        fingerprints = torch.cat(fingerprints)
        adjacencies = self.pad(adjacencies, 0)

        """GNN layer (update the fingerprint vectors)."""
        fingerprint_vectors = self.embed_fingerprint(fingerprints)
        for l in range(self.layer_hidden):
            hs = self.update(adjacencies, fingerprint_vectors, l)
            fingerprint_vectors = F.normalize(hs, 2, 1)  # normalize.

        """Molecular vector by sum or mean of the fingerprint vectors."""
        molecular_vectors = self.sum(fingerprint_vectors, molecular_sizes)
        # molecular_vectors = self.mean(fingerprint_vectors, molecular_sizes)

        return molecular_vectors

    def mlp(self, vectors):
        """Classifier or regressor based on multilayer perceptron."""
        for l in range(self.layer_output):
            vectors = torch.relu(self.W_output[l](vectors))
        outputs = self.W_property(vectors)
        return outputs

    def forward_classifier(self, data_batch, train):

        inputs = data_batch[:-1]
        correct_labels = torch.cat(data_batch[-1])

        if train:
            molecular_vectors = self.gnn(inputs)
            predicted_scores = self.mlp(molecular_vectors)
            loss = F.cross_entropy(predicted_scores, correct_labels)
            return loss
        else:
            with torch.no_grad():
                molecular_vectors = self.gnn(inputs)
                predicted_scores = self.mlp(molecular_vectors)
            predicted_scores = predicted_scores.to('cpu').data.numpy()
            predicted_scores = [s[1] for s in predicted_scores]
            correct_labels = correct_labels.to('cpu').data.numpy()
            return predicted_scores, correct_labels

    def forward_regressor(self, data_batch, train):

        inputs = data_batch[:-1]
        correct_values = torch.cat(data_batch[-1])

        if train:
            molecular_vectors = self.gnn(inputs)
            predicted_values = self.mlp(molecular_vectors)
            loss = F.mse_loss(predicted_values, correct_values)
            return loss
        else:
            with torch.no_grad():
                molecular_vectors = self.gnn(inputs)
                predicted_values = self.mlp(molecular_vectors)
            predicted_values = predicted_values.to('cpu').data.numpy()
            correct_values = correct_values.to('cpu').data.numpy()
            predicted_values = np.concatenate(predicted_values)
            correct_values = np.concatenate(correct_values)
            return predicted_values, correct_values


class Trainer(object):
    def __init__(self, model):
        self.model = model
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def train(self, dataset):
        np.random.shuffle(dataset)
        N = len(dataset)
        loss_total = 0
        for i in range(0, N, batch_train):
            data_batch = list(zip(*dataset[i:i+batch_train]))
            if task == 'classification':
                loss = self.model.forward_classifier(data_batch, train=True)
            if task == 'regression':
                loss = self.model.forward_regressor(data_batch, train=True)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            loss_total += loss.item()
        return loss_total


class Tester(object):
    def __init__(self, model):
        self.model = model

    def test_classifier(self, dataset):
        N = len(dataset)
        P, C = [], []
        for i in range(0, N, batch_test):
            data_batch = list(zip(*dataset[i:i+batch_test]))
            predicted_scores, correct_labels = self.model.forward_classifier(
                                               data_batch, train=False)
            P.append(predicted_scores)
            C.append(correct_labels)
        AUC = roc_auc_score(np.concatenate(C), np.concatenate(P))
        return AUC

    def test_regressor(self, dataset):
        N = len(dataset)
        SAE = 0  # sum absolute error.
        for i in range(0, N, batch_test):
            data_batch = list(zip(*dataset[i:i+batch_test]))
            predicted_values, correct_values = self.model.forward_regressor(
                                               data_batch, train=False)
            SAE += sum(np.abs(predicted_values-correct_values))
        MAE = SAE / N  # mean absolute error.
        return MAE
    
    def predict_regressor(self, dataset):
        N = len(dataset)
        predictions = []
        for i in range(0, N, 1):
            data_batch = list(zip(*dataset[i:i+1]))
            predicted_values, correct_values = self.model.forward_regressor(
                                               data_batch, train=False)
            predictions.append(predicted_values)
        return predictions

    def save_result(self, result, filename):
        with open(filename, 'a') as f:
            f.write(result + '\n')

def create_atoms(mol, atom_dict):
    """Transform the atom types in a molecule (e.g., H, C, and O)
    into the indices (e.g., H=0, C=1, and O=2).
    Note that each atom index considers the aromaticity.
    """
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    for a in mol.GetAromaticAtoms():
        i = a.GetIdx()
        atoms[i] = (atoms[i], 'aromatic')
    atoms = [atom_dict[a] for a in atoms]
    return np.array(atoms)


def create_ijbonddict(mol, bond_dict):
    """Create a dictionary, in which each key is a node ID
    and each value is the tuples of its neighboring node
    and chemical bond (e.g., single and double) IDs.
    """
    i_jbond_dict = defaultdict(lambda: [])
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bond = bond_dict[str(b.GetBondType())]
        i_jbond_dict[i].append((j, bond))
        i_jbond_dict[j].append((i, bond))
    return i_jbond_dict


def extract_fingerprints(radius, atoms, i_jbond_dict,
                         fingerprint_dict, edge_dict):
    """Extract the fingerprints from a molecular graph
    based on Weisfeiler-Lehman algorithm.
    """

    if (len(atoms) == 1) or (radius == 0):
        nodes = [fingerprint_dict[a] for a in atoms]

    else:
        nodes = atoms
        i_jedge_dict = i_jbond_dict

        for _ in range(radius):

            """Update each node ID considering its neighboring nodes and edges.
            The updated node IDs are the fingerprint IDs.
            """
            nodes_ = []
            for i, j_edge in i_jedge_dict.items():
                neighbors = [(nodes[j], edge) for j, edge in j_edge]
                fingerprint = (nodes[i], tuple(sorted(neighbors)))
                if fingerprint not in fingerprint_dict:
                    fingerprint_dict[fingerprint] = len(fingerprint_dict)
                nodes_.append(fingerprint_dict[fingerprint])

            """Also update each edge ID considering
            its two nodes on both sides.
            """
            i_jedge_dict_ = defaultdict(lambda: [])
            for i, j_edge in i_jedge_dict.items():
                for j, edge in j_edge:
                    both_side = tuple(sorted((nodes[i], nodes[j])))
                    edge = edge_dict[(both_side, edge)]
                    i_jedge_dict_[i].append((j, edge))

            nodes = nodes_
            i_jedge_dict = i_jedge_dict_

    return np.array(nodes)

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('The code uses a GPU!')
else:
    device = torch.device('cpu')
    print('The code uses a CPU...')

device = torch.device('cpu')
def predict(lab, smiles_list, model_path, dict_path):
    n_fingerprints = nips_gnn[lab]['n_fingerprints']
    dim = nips_gnn[lab]['dim']
    layer_hidden = nips_gnn[lab]['layer_hidden']
    layer_output = nips_gnn[lab]['layer_output']
    radius = nips_gnn[lab]['radius']
    
    # Reinitialize the model structure
    model = MolecularGraphNeuralNetwork(n_fingerprints, dim, layer_hidden, layer_output).to(device)
    
    # Load the saved state_dict
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    # Load the dictionaries used during training
    with open(dict_path, 'rb') as f:
        atom_dict, bond_dict, fingerprint_dict, edge_dict = pickle.load(f)

    predictions = []
    tester = Tester(model)
    failed = 0
    failed_smiles = []
    for smiles in tqdm(smiles_list, total=len(smiles_list), desc=f"Predicting SMILES"):
        try:
            mol = Chem.MolFromSmiles(smiles)
            smiles = Chem.MolToSmiles(mol, canonical=True)
            mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
            atoms = create_atoms(mol, atom_dict)
            molecular_size = len(atoms)
            i_jbond_dict = create_ijbonddict(mol, bond_dict)
            fingerprints = extract_fingerprints(radius, atoms, i_jbond_dict,
                                                   fingerprint_dict, edge_dict)
            adjacency = Chem.GetAdjacencyMatrix(mol)
    
            # Convert to PyTorch tensors
            fingerprints = torch.LongTensor(fingerprints).to(device)
            adjacency = torch.FloatTensor(adjacency).to(device)
    
            dataset = [(fingerprints, adjacency, molecular_size, torch.FloatTensor([[float(0)]]).to(device))]
            predictions.append(tester.predict_regressor(dataset))
        except Exception as e:
            print(e)
            failed += 1
            failed_smiles += [smiles]
            predictions.append(None)

    if failed != 0:
        model_dir = f'/kaggle/input/neurips-chembert-with-augmented-data/transformers/default/29/GNN_Models/GNN_Models/{lab}_GNN_Model/2 layers'
        tmp_df = pd.DataFrame({
            'SMILES': failed_smiles
        })
        tmp_df.to_csv("/kaggle/working/smiles.csv", index=False)
        pred = pred_on_DA_GNN(lab, model_dir, f"/kaggle/working/smiles.csv")
        i = 0
        j = 0
        while (i < len(predictions)):
            if predictions[i] == None:
                predictions[i] = [pred[j],]
                j += 1
            i += 1
        print(j, len(pred))
    print(f"Failed SMILES: {failed}")
    return predictions

task = 'regression'
nips_gnn = {
    'Tg' : {
        'radius': 1,
        'dim': 50,
        'layer_hidden': 6,
        'layer_output': 6,
        'n_fingerprints': 651,
        'model_path': "/kaggle/input/neurips-chembert-with-augmented-data/pytorch/nips-e50/4/model--Tg--radius1--dim50--layer_hidden6--layer_output6--batch_train32--batch_test32--lr1e-4--lr_decay0.99--decay_interval10--weight_decay1e-6--iteration14.pt"
    },
    'FFV' : {
        'radius': 1,
        'dim': 100,
        'layer_hidden': 12,
        'layer_output': 12,
        'n_fingerprints': 549,
        'model_path': "/kaggle/input/neurips-chembert-with-augmented-data/pytorch/nips-e50/4/model--FFV--radius1--dim100--layer_hidden12--layer_output12--batch_train16--batch_test16--lr1e-5--lr_decay0.99--decay_interval10--weight_decay1e-6--iteration100.pt",
    },
}

# All Properties Inference
---

In [ ]:
test_df = pd.read_csv("/kaggle/input/neurips-polymer/input.csv")
labels = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']

for label in ['Density', 'Rg']:
    test_smiles = test_df['SMILES'].tolist()
    
    fingerprints, descriptors, valid_smiles, invalid_indices = smiles_to_combined_fingerprints_with_descriptors(test_smiles, radius=2, n_bits=128)
    
    X = pd.DataFrame(descriptors)
    X = X.drop(['BCUT2D_MWLOW','BCUT2D_MWHI','BCUT2D_CHGHI','BCUT2D_CHGLO','BCUT2D_LOGPHI','BCUT2D_LOGPLOW','BCUT2D_MRLOW','BCUT2D_MRHI','MinAbsPartialCharge','MaxPartialCharge','MinPartialCharge','MaxAbsPartialCharge',],axis=1)
    selected = filters[label]   # if filters[label] is already a list
    X = X.filter(items=selected)

    fp_df = pd.DataFrame(fingerprints, columns=[f'FP_{i}' for i in range(fingerprints.shape[1])])    
    print(fp_df.shape)

    fp_df.reset_index(drop=True, inplace=True)
    X.reset_index(drop=True, inplace=True)
    X = pd.concat([X, fp_df], axis=1)

    print(f"After concat: {X.shape}")
    test_data = X
    test_data.to_csv(f"{label}_test.csv", index=False)

In [ ]:
pred_df = {}
for label in ['Density', 'Rg']:
    print(f"\nPredicting target {label} ...")
    host_csv = f"/kaggle/working/{label}_test.csv"
    preds, allp = eval_on_host_train(label, host_csv, model_pattern=f"model_{label}_fold*.pt", desc_cols_file=f"desc_cols_{label}.pkl")
    pred_df[label] = preds
    print(f" -> Completed {label}: {preds}")

for i in ['Tc']:
    print(f"\nPredicting target {i} ...")
    model_dir = f'/kaggle/input/neurips-chembert-with-augmented-data/transformers/default/29/GNN_Models/GNN_Models/{i}_GNN_Model/2 layers'
    preds = pred_on_DA_GNN(i, model_dir, f"/kaggle/input/neurips-polymer/input.csv")
    pred_df[i] = preds
    print(f" -> Completed {i}: {preds}")

for lab in ['FFV', 'Tg']:
    print(f"\nPredicting target {lab} ...")
    dict_path = f'/kaggle/input/neurips-chembert-with-augmented-data/pytorch/nips-e50/4/{lab.lower()}_dictionaries.pkl'
    model_path = nips_gnn[lab]['model_path']
    
    device = torch.device('cpu')
    smiles_list = pd.read_csv(f"/kaggle/input/neurips-polymer/input.csv")['SMILES'].to_list()
    predictions = predict(lab, smiles_list, model_path, dict_path)
    for i in range(len(predictions)):
        predictions[i] = predictions[i][0][0]
    pred_df[lab] = predictions
    print(f" -> Completed {lab}: {predictions}")

In [ ]:
submission_df = pd.DataFrame({
    'SMILES': test_df['SMILES'].to_list()
})

for i in labels:
    submission_df[i] = pred_df[i]

In [ ]:
!rm -r /kaggle/working/*
!rm /kaggle/working/*

In [ ]:
submission_df_path = "submission.csv"
submission_df.to_csv("submission.csv", index=False)
print(submission_df)

---